In [1]:
import pandas as pd
import numpy as np


In [2]:
df=pd.read_csv('../data/interim/listings_parsed_v7.csv')

In [3]:
df.columns

Index(['Listing Title', 'area_m2', 'Distances to Railway Stations',
       'Building Age', 'Stories Tall', 'Floor', 'Price',
       'Administrative Costs', 'Deposit', 'Gratuity', 'Rooms', 'Square Meters',
       'Links to Listing', 'jnc', 'Price_yen', 'Administrative Costs_yen',
       'Deposit_yen', 'Gratuity_yen', 'Admin_yen', 'deposit_months',
       'gratuity_months', 'lookalike_id', 'stories_above', 'stories_below',
       'floor_min', 'floor_max', 'is_multi_floor', 'Building_Age_Parsed',
       'age_years', 'is_new', 'age_is_lower_bound', 'is_hiraya',
       'flag_floor_above_building', 'flag_area_small', 'flag_area_large',
       'flag_fee_tiny', 'no_gratuity', 'n_rooms', 'has_L', 'has_D', 'has_K',
       'has_S', 'is_1R', 'area_per_room', 'flag_rooms_implausible',
       'flag_layout_area_conflict', 'walk_min_nearest', 'walk_min_second',
       'n_walk_stations', 'n_lines', 'walk_le_5', 'walk_le_10', 'walk_le_15',
       'has_bus_access', 'has_car_access', 'location', 'municipa

In [4]:


import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

splits = list(GroupKFold(n_splits=5).split(df, df["Price_yen"], groups=df["group_coarse"]))
base = ["area_m2", "age_years", "floor_min", "stories_above", "walk_min_nearest"]
cat = ["municipality"]
groups = {"madori": ["n_rooms","has_L","has_D","has_K","has_S","is_1R"]}

def make_pipe(num_cols):
    return Pipeline([
        ("preprocessor", ColumnTransformer([
            ("num", "passthrough", num_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat)
        ])),
        ("regressor", XGBRegressor(n_estimators=400, learning_rate=0.1, max_depth=6,
                                   tree_method="hist", device="cuda", random_state=0, n_jobs=-1))
    ])

def fold_maes(num_cols):
    maes = []
    for tr, te in splits:
        pipe = make_pipe(num_cols)
        pipe.fit(df.iloc[tr][num_cols + cat], df.iloc[tr]["Price_yen"])
        preds = pipe.predict(df.iloc[te][num_cols + cat])
        maes.append(mean_absolute_error(df.iloc[te]["Price_yen"], preds))
    return np.array(maes)

base_maes = fold_maes(base)

with_madori_maes = fold_maes(base + groups["madori"])

diff = with_madori_maes - base_maes

print(f"Base mean MAE: {base_maes.mean():,.2f}")
print(f"With madori mean MAE: {with_madori_maes.mean():,.2f}")
print(f"Mean difference: {diff.mean():,.2f}")
print(f"Difference std: {diff.std():,.2f}")
print(f"Folds improved: {(diff < 0).sum()} / {len(diff)}")


c:\Users\priya\Desktop\data science try\tokyo-rental-intelligence\.venv\Lib\site-packages\xgboost\core.py:569: UserWarning: [14:00:17] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Base mean MAE: 11,302.69
With madori mean MAE: 11,179.41
Mean difference: -123.28
Difference std: 94.34
Folds improved: 5 / 5


In [5]:
df["age_ge_45"] = (df["age_years"] >= 45).astype(int)
splits = list(GroupKFold(n_splits=5).split(df, df["Price_yen"], groups=df["group_coarse"]))
base = ["area_m2", "age_years", "floor_min", "stories_above", "walk_min_nearest"]
cat = ["municipality"]
groups = {
    "is_new": ["is_new"],
    "madori": ["n_rooms","has_L","has_D","has_K","has_S","is_1R"],
    "station": ["walk_min_second","n_lines","has_bus_access"],
    "age_ge_45": ["age_ge_45"],
    "extras": ["is_multi_floor","is_hiraya","stories_below"],
}

def make_pipe(num_cols):
    return Pipeline([
        ("preprocessor", ColumnTransformer([
            ("num", "passthrough", num_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat)
        ])),
        ("regressor", XGBRegressor(n_estimators=400, learning_rate=0.1, max_depth=6,
                                   tree_method="hist", device="cuda", random_state=0, n_jobs=-1))
    ])

def fold_maes(num_cols):
    maes = []
    for tr, te in splits:
        pipe = make_pipe(num_cols)
        pipe.fit(df.iloc[tr][num_cols + cat], df.iloc[tr]["Price_yen"])
        preds = pipe.predict(df.iloc[te][num_cols + cat])
        maes.append(mean_absolute_error(df.iloc[te]["Price_yen"], preds))
    return np.array(maes)

results = {}
for name, cols in groups.items():
    d = fold_maes(base + cols) - base_maes
    results[name] = d
    print(f"{name:10s} diff {d.mean():8.1f} ± {d.std():6.1f}  improved {(d<0).sum()}/5")


is_new     diff     17.1 ±   33.9  improved 2/5
madori     diff   -123.3 ±   94.3  improved 5/5
station    diff    -54.3 ±  160.9  improved 3/5
age_ge_45  diff      0.0 ±    0.0  improved 0/5
extras     diff    100.6 ±  128.0  improved 2/5


In [6]:
df[["is_new", "age_ge_45"]].describe()

,is_new,age_ge_45
count,183490.000000,183490.000000
mean,0.157845,0.039986
std,0.364597,0.195926
min,0.000000,0.000000
25%,0.000000,0.000000
50%,0.000000,0.000000
75%,0.000000,0.000000
max,1.000000,1.000000


In [7]:
# H1 and H4: ablate nonlinear terms on the log-area + log-target linear model, GroupKFold paired folds.
# Imports: numpy as np, pandas as pd, GroupKFold, mean_absolute_error, ColumnTransformer,
#   Pipeline, SimpleImputer, StandardScaler, OneHotEncoder, SplineTransformer,
#   FunctionTransformer, LinearRegression, TransformedTargetRegressor.
#
# Reuse the SAME splits as the XGBoost ablation: splits = list(GroupKFold(5).split(df, df["Price_yen"], groups=df["group_coarse"])).
# Create df["age_ge_45"] = (df["age_years"] >= 45).astype(int) if it does not exist.
#
# def make_linear(walk_mode, age_mode, extra_cols=[]): return a TransformedTargetRegressor(
#     regressor=Pipeline([("pre", ColumnTransformer([...])), ("lr", LinearRegression())]),
#     func=np.log1p, inverse_func=lambda z: np.clip(np.expm1(z), 1e4, 5e6))
# ColumnTransformer pieces (every learned step inside the pipeline):
#   ("area", Pipeline([("log", FunctionTransformer(np.log)), ("sc", StandardScaler())]), ["area_m2"])
#   ("fl", Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True)), ("sc", StandardScaler())]),
#       ["floor_min", "stories_above"])
#   ("cat", OneHotEncoder(handle_unknown="ignore"), ["municipality"])
#   walk_mode "raw":    ("walk", Pipeline([imputer median + add_indicator, StandardScaler]), ["walk_min_nearest"])
#   walk_mode "spline": ("walk", Pipeline([imputer median + add_indicator, SplineTransformer(n_knots=6, degree=3, knots="quantile")]), ["walk_min_nearest"])
#   walk_mode "flags":  ("walk", Pipeline([imputer median + add_indicator, StandardScaler]), ["walk_min_nearest"]) plus passthrough of ["walk_le_5","walk_le_10","walk_le_15"] (fill NaN with 0)
#   age_mode "raw":     age_years -> StandardScaler
#   age_mode "spline":  age_years -> SplineTransformer(n_knots=7, degree=3, knots="quantile")
#   extra_cols: passthrough columns such as ["age_ge_45", "is_new"].
#
# Runs (each returns 5 fold MAEs on raw yen, fit on train rows only, no tuning):
#   A: walk raw,    age raw                    <- base
#   B: walk spline, age raw                    (H1)
#   C: walk flags,  age raw                    (H1, threshold version)
#   D: walk raw,    age spline                 (age nonlinearity)
#   E: walk raw,    age raw,    extra ["age_ge_45"]            (H4 on a linear age term)
#   F: walk raw,    age spline, extra ["age_ge_45"]            (H4 on top of a flexible age term)
# Print for each run vs A: mean MAE, mean diff, diff std, folds improved.
# Also print MAE within the age_years >= 45 segment only for D and F (H4 is low-powered overall).
# Do not modify splits. Do not tune.

In [8]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, SplineTransformer, FunctionTransformer
from sklearn.linear_model import LinearRegression
from sklearn.compose import TransformedTargetRegressor

if "age_ge_45" not in df.columns:
    df["age_ge_45"] = (df["age_years"] >= 45).astype(int)


def make_linear(walk_mode, age_mode, extra_cols=None):
    extra_cols = extra_cols or []

    transformers = [
        ("area", Pipeline([
            ("log", FunctionTransformer(np.log)),
            ("sc", StandardScaler())
        ]), ["area_m2"]),

        ("fl", Pipeline([
            ("imp", SimpleImputer(strategy="median", add_indicator=True)),
            ("sc", StandardScaler())
        ]), ["floor_min", "stories_above"]),

        ("cat", OneHotEncoder(handle_unknown="ignore"), ["municipality"]),
    ]

    if walk_mode == "raw":
        transformers.append((
            "walk",
            Pipeline([
                ("imp", SimpleImputer(strategy="median", add_indicator=True)),
                ("sc", StandardScaler())
            ]),
            ["walk_min_nearest"]
        ))
    elif walk_mode == "spline":
        transformers.append((
            "walk",
            Pipeline([
                ("imp", SimpleImputer(strategy="median", add_indicator=True)),
                ("spline", SplineTransformer(
                    n_knots=6, degree=3, knots="quantile"
                ))
            ]),
            ["walk_min_nearest"]
        ))
    elif walk_mode == "flags":
        transformers.extend([
            (
                "walk",
                Pipeline([
                    ("imp", SimpleImputer(strategy="median", add_indicator=True)),
                    ("sc", StandardScaler())
                ]),
                ["walk_min_nearest"]
            ),
            (
                "walk_flags",
                SimpleImputer(strategy="constant", fill_value=0),
                ["walk_le_5", "walk_le_10", "walk_le_15"]
            )
        ])

    if age_mode == "raw":
        transformers.append((
            "age",
            Pipeline([
                ("imp", SimpleImputer(strategy="median")),
                ("sc", StandardScaler())
            ]),
            ["age_years"]
        ))
    elif age_mode == "spline":
        transformers.append((
            "age",
            Pipeline([
                ("imp", SimpleImputer(strategy="median")),
                ("spline", SplineTransformer(
                    n_knots=7, degree=3, knots="quantile"
                ))
            ]),
            ["age_years"]
        ))

    if extra_cols:
        transformers.append((
            "extra",
            SimpleImputer(strategy="constant", fill_value=0),
            extra_cols
        ))

    feature_cols = list(dict.fromkeys(
        col for _, _, cols in transformers for col in cols
    ))

    return TransformedTargetRegressor(
        regressor=Pipeline([
            ("pre", ColumnTransformer(transformers)),
            ("lr", LinearRegression())
        ]),
        func=np.log1p,
        inverse_func=lambda z: np.clip(np.expm1(z), 1e4, 5e6)
    ), feature_cols


runs = {
    "A": ("raw", "raw", []),
    "B": ("spline", "raw", []),
    "C": ("flags", "raw", []),
    "D": ("raw", "spline", []),
    "E": ("raw", "raw", ["age_ge_45"]),
    "F": ("raw", "spline", ["age_ge_45"]),
}

run_maes = {}
segment_maes = {}

for name, (walk_mode, age_mode, extra_cols) in runs.items():
    fold_maes = []
    fold_segment_maes = []

    model, feature_cols = make_linear(walk_mode, age_mode, extra_cols)

    for tr, te in splits:
        model.fit(df.iloc[tr][feature_cols], df.iloc[tr]["Price_yen"])
        preds = model.predict(df.iloc[te][feature_cols])

        y_test = df.iloc[te]["Price_yen"].to_numpy()
        fold_maes.append(mean_absolute_error(y_test, preds))

        if name in {"D", "F"}:
            mask = df.iloc[te]["age_years"].to_numpy() >= 45
            fold_segment_maes.append(
                mean_absolute_error(y_test[mask], preds[mask])
            )

    run_maes[name] = np.array(fold_maes)

    if name in {"D", "F"}:
        segment_maes[name] = np.array(fold_segment_maes)


a_maes = run_maes["A"]

for name, maes in run_maes.items():
    diff = maes - a_maes
    print(
        f"{name}: MAE {maes.mean():,.2f}, "
        f"mean diff {diff.mean():,.2f}, "
        f"diff std {diff.std():,.2f}, "
        f"improved {(diff < 0).sum()}/5"
    )

for name in ["D", "F"]:
    print(
        f"{name} age>=45 MAE: "
        f"{segment_maes[name].mean():,.2f}"
    )

A: MAE 12,924.15, mean diff 0.00, diff std 0.00, improved 0/5
B: MAE 12,897.65, mean diff -26.50, diff std 17.17, improved 5/5
C: MAE 12,909.98, mean diff -14.17, diff std 15.11, improved 3/5
D: MAE 12,765.32, mean diff -158.83, diff std 11.66, improved 5/5
E: MAE 12,863.38, mean diff -60.77, diff std 8.54, improved 5/5
F: MAE 12,765.75, mean diff -158.40, diff std 11.81, improved 5/5
D age>=45 MAE: 11,592.86
F age>=45 MAE: 11,590.71


In [9]:

FEATURES = [
    "area_m2", "age_years", "floor_min", "stories_above",
    "walk_min_nearest", "n_rooms", "has_L", "has_D", "has_K", "has_S", "is_1R"
]
CAT = ["municipality"]

oof = pd.Series(np.nan, index=df.index, dtype=float)
df["fold"] = np.nan

for fold_id, (tr, te) in enumerate(splits):
    pipe = make_pipe(FEATURES)
    train_cols = FEATURES + CAT

    pipe.fit(
        df.iloc[tr][train_cols],
        df.iloc[tr]["Price_yen"]
    )
    oof.iloc[te] = pipe.predict(df.iloc[te][train_cols])
    df.iloc[te, df.columns.get_loc("fold")] = fold_id

assert oof.isna().sum() == 0

abs_err = (oof - df["Price_yen"]).abs()
pct_err = abs_err / df["Price_yen"]

print(f"Overall MAE: {abs_err.mean():,.2f}")


def error_table(group):
    return (
        pd.DataFrame({
            "group": group,
            "abs_err": abs_err,
            "pct_err": pct_err,
        })
        .groupby("group", dropna=False)
        .agg(
            n=("abs_err", "size"),
            MAE=("abs_err", "mean"),
            median_abs_error=("abs_err", "median"),
            median_pct_error=("pct_err", "median"),
        )
        .reset_index()
    )


price_band = pd.cut(
    df["Price_yen"],
    [0, 80_000, 150_000, np.inf],
    labels=["<80k", "80-150k", ">150k"]
)

if "region" in df.columns:
    region = df["region"].fillna("unknown")
else:
    municipality = df["municipality"].astype("string")
    region = pd.Series(
        np.select(
            [
                municipality.str.endswith("区", na=False),
                municipality.str.endswith("市", na=False),
                municipality.str.endswith(("町", "村"), na=False),
            ],
            ["ward", "city", "town_village"],
            default="unknown",
        ),
        index=df.index,
    )

area_band = pd.cut(
    df["area_m2"],
    [0, 20, 30, 50, 80, np.inf],
    labels=["<20", "20-30", "30-50", "50-80", ">80"]
)

print("\nPrice bands")
display(error_table(price_band))

print("\nRegions")
display(error_table(region))

print("\nArea bands")
display(error_table(area_band))

output_path = "../data/interim/oof_xgb_groupkfold.csv"
oof.rename("oof_prediction").to_csv(output_path, index=True)
print(f"\nSaved OOF predictions to {output_path}")



Overall MAE: 11,179.41

Price bands


,group,n,MAE,median_abs_error,median_pct_error
0,<80k,88346,5983.820677,4778.847656,0.080524
1,80-150k,74126,10528.769978,8112.328125,0.078989
2,>150k,21018,35312.999162,20155.460938,0.098709



Regions


,group,n,MAE,median_abs_error,median_pct_error
0,city,59320,7400.592017,5201.470703,0.084243
1,town_village,317,6409.325907,3526.406250,0.076264
2,ward,123853,13001.509584,7511.039062,0.080707



Area bands


,group,n,MAE,median_abs_error,median_pct_error
0,<20,35424,5541.163806,4412.919922,0.083866
1,20-30,67608,7989.017870,6087.630859,0.077482
2,30-50,49644,10539.086331,7455.281250,0.078896
3,50-80,26009,17927.752654,11409.984375,0.092923
4,>80,4805,67723.998735,29291.250000,0.127067



Saved OOF predictions to ../data/interim/oof_xgb_groupkfold.csv


In [10]:
FEATURES = [
    "area_m2", "age_years", "floor_min", "stories_above",
    "walk_min_nearest", "n_rooms", "has_L", "has_D",
    "has_K", "has_S", "is_1R"
]
CAT = ["municipality"]
train_cols = FEATURES + CAT

def make_q_pipe(alpha):
    return Pipeline([
        ("preprocessor", ColumnTransformer([
            ("num", "passthrough", FEATURES),
            ("cat", OneHotEncoder(handle_unknown="ignore"), CAT),
        ])),
        ("regressor", XGBRegressor(
            objective="reg:quantileerror",
            quantile_alpha=alpha,
            n_estimators=400,
            learning_rate=0.05,
            max_depth=6,
            tree_method="hist",
            device="cuda",
            random_state=0,
            n_jobs=-1,
        )),
    ])

intervals = pd.DataFrame(
    np.nan,
    index=df.index,
    columns=["lo", "mid", "hi"],
)
intervals["fold"] = np.nan

z = np.log1p(df["Price_yen"])

for fold_id, (tr, te) in enumerate(splits):
    X_train = df.iloc[tr][train_cols]
    X_test = df.iloc[te][train_cols]
    y_train = z.iloc[tr]

    for alpha, column in [(0.1, "lo"), (0.5, "mid"), (0.9, "hi")]:
        q_pipe = make_q_pipe(alpha)
        q_pipe.fit(X_train, y_train)

        predictions = np.expm1(q_pipe.predict(X_test))
        intervals.loc[df.iloc[te].index, column] = predictions

    intervals.loc[df.iloc[te].index, "fold"] = fold_id

assert intervals[["lo", "mid", "hi"]].notna().all().all()

intervals["lo"] = np.minimum(intervals["lo"], intervals["mid"])
intervals["hi"] = np.maximum(intervals["hi"], intervals["mid"])

y = df["Price_yen"]
inside = y.between(intervals["lo"], intervals["hi"])
width_pct = (intervals["hi"] - intervals["lo"]) / intervals["mid"]

print(f"Coverage: {inside.mean():.4f} (target 0.8000)")
print(f"Mean width (%): {width_pct.mean():.4f}")
print(f"Median width (%): {width_pct.median():.4f}")
print(f"Median MAE: {(intervals['mid'] - y).abs().mean():,.2f}")


def interval_summary(group):
    return (
        pd.DataFrame({
            "group": group,
            "inside": inside,
            "width_pct": width_pct,
        })
        .groupby("group", dropna=False)
        .agg(
            coverage=("inside", "mean"),
            median_width_pct=("width_pct", "median"),
            n=("inside", "size"),
        )
        .reset_index()
    )


price_band = pd.cut(
    y,
    [0, 80_000, 150_000, np.inf],
    labels=["<80k", "80-150k", ">150k"],
)

area_band = pd.cut(
    df["area_m2"],
    [0, 20, 30, 50, 80, np.inf],
    labels=["<20", "20-30", "30-50", "50-80", ">80"],
)

print("\nPrice bands")
display(interval_summary(price_band))

print("\nArea bands")
display(interval_summary(area_band))

print("\nRegions")
display(interval_summary(df["region"].fillna("unknown")))

print("\nFolds")
display(interval_summary(intervals["fold"].astype("Int64")))

interval_output_path = "../data/interim/quantile_intervals_groupkfold.csv"
intervals[["lo", "mid", "hi"]].to_csv(interval_output_path, index=True)

print(f"\nSaved intervals to {interval_output_path}")

Coverage: 0.7989 (target 0.8000)
Mean width (%): 0.4132
Median width (%): 0.3982
Median MAE: 13,037.04

Price bands


,group,coverage,median_width_pct,n
0,<80k,0.792305,0.398967,88346
1,80-150k,0.839287,0.394064,74126
2,>150k,0.684271,0.416052,21018



Area bands


,group,coverage,median_width_pct,n
0,<20,0.799373,0.422976,35424
1,20-30,0.800556,0.363403,67608
2,30-50,0.803440,0.411727,49644
3,50-80,0.795455,0.477776,26009
4,>80,0.744225,0.548855,4805



Regions


,group,coverage,median_width_pct,n
0,city,0.766605,0.427522,59320
1,town_village,0.287066,0.576784,317
2,ward,0.815693,0.386568,123853



Folds


,group,coverage,median_width_pct,n
0,0,0.796665,0.402163,36698
1,1,0.799063,0.400608,36698
2,2,0.789689,0.394363,36698
3,3,0.807455,0.395649,36698
4,4,0.801679,0.398101,36698



Saved intervals to ../data/interim/quantile_intervals_groupkfold.csv


In [12]:
from sklearn.model_selection import GroupShuffleSplit

train_cols = FEATURES + CAT
z_all = np.log1p(df["Price_yen"])

calibrated = pd.DataFrame(
    np.nan,
    index=df.index,
    columns=["lo", "mid", "hi", "fold"],
)

for fold_id, (tr, te) in enumerate(splits):
    outer_labels = df.iloc[tr].index

    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
    fit_pos, cal_pos = next(
        gss.split(
            np.zeros(len(tr)),
            groups=df.iloc[tr]["group_coarse"].to_numpy(),
        )
    )

    fit_idx = outer_labels[fit_pos]
    cal_idx = outer_labels[cal_pos]
    test_idx = df.iloc[te].index

    fitted = {}
    for alpha in [0.1, 0.5, 0.9]:
        model = make_q_pipe(alpha)
        model.fit(
            df.loc[fit_idx, train_cols],
            z_all.loc[fit_idx],
        )
        fitted[alpha] = model

    cal_lo = fitted[0.1].predict(df.loc[cal_idx, train_cols])
    cal_mid = fitted[0.5].predict(df.loc[cal_idx, train_cols])
    cal_hi = fitted[0.9].predict(df.loc[cal_idx, train_cols])

    cal_lo = np.minimum(cal_lo, cal_mid)
    cal_hi = np.maximum(cal_hi, cal_mid)

    cal_z = z_all.loc[cal_idx].to_numpy()
    scores = np.maximum(cal_lo - cal_z, cal_z - cal_hi)

    def conformal_quantile(values):
        n = len(values)
        level = min(1.0, np.ceil((n + 1) * 0.8) / n)
        return np.quantile(values, level, method="higher")

    q_global = conformal_quantile(scores)

    cal_mid_yen = np.expm1(cal_mid)
    cal_bands = pd.cut(
        cal_mid_yen,
        [0, 80_000, 150_000, np.inf],
        labels=["<80k", "80-150k", ">150k"],
    )

    q_by_band = {}
    for band in cal_bands.categories:
        band_scores = scores[cal_bands.to_numpy() == band]
        q_by_band[band] = (
            conformal_quantile(band_scores)
            if len(band_scores) >= 200
            else q_global
        )

    test_lo = fitted[0.1].predict(df.loc[test_idx, train_cols])
    test_mid = fitted[0.5].predict(df.loc[test_idx, train_cols])
    test_hi = fitted[0.9].predict(df.loc[test_idx, train_cols])

    test_lo = np.minimum(test_lo, test_mid)
    test_hi = np.maximum(test_hi, test_mid)

    test_bands = pd.cut(
        np.expm1(test_mid),
        [0, 80_000, 150_000, np.inf],
        labels=["<80k", "80-150k", ">150k"],
    )

    q_test = np.array([
        q_by_band.get(band, q_global)
        for band in test_bands
    ])

    calibrated.loc[test_idx, "lo"] = np.expm1(test_lo - q_test)
    calibrated.loc[test_idx, "mid"] = np.expm1(test_mid)
    calibrated.loc[test_idx, "hi"] = np.expm1(test_hi + q_test)
    calibrated.loc[test_idx, "fold"] = fold_id

assert calibrated[["lo", "mid", "hi", "fold"]].notna().all().all()

cal_inside = y.between(calibrated["lo"], calibrated["hi"])
cal_width_pct = (
    calibrated["hi"] - calibrated["lo"]
) / calibrated["mid"]

uncal_inside = y.between(intervals["lo"], intervals["hi"])
uncal_width_pct = (
    intervals["hi"] - intervals["lo"]
) / intervals["mid"]

print("Overall")
print(
    f"Calibrated:   coverage={cal_inside.mean():.4f}, "
    f"median width={cal_width_pct.median():.4f}"
)
print(
    f"Uncalibrated: coverage={uncal_inside.mean():.4f}, "
    f"median width={uncal_width_pct.median():.4f}"
)


def compare_summary(group):
    return (
        pd.DataFrame({
            "group": group,
            "cal_inside": cal_inside,
            "cal_width_pct": cal_width_pct,
            "uncal_inside": uncal_inside,
            "uncal_width_pct": uncal_width_pct,
        })
        .groupby("group", dropna=False)
        .agg(
            calibrated_coverage=("cal_inside", "mean"),
            calibrated_width_pct=("cal_width_pct", "median"),
            uncalibrated_coverage=("uncal_inside", "mean"),
            uncalibrated_width_pct=("uncal_width_pct", "median"),
            n=("cal_inside", "size"),
        )
        .reset_index()
    )


true_price_band = pd.cut(
    y,
    [0, 80_000, 150_000, np.inf],
    labels=["<80k", "80-150k", ">150k"],
)

predicted_price_band = pd.cut(
    calibrated["mid"],
    [0, 80_000, 150_000, np.inf],
    labels=["<80k", "80-150k", ">150k"],
)

print("\nTRUE price band")
display(compare_summary(true_price_band))

print("\nPREDICTED price band")
display(compare_summary(predicted_price_band))

print("\nArea band")
display(compare_summary(area_band))

print("\nRegion")
display(compare_summary(region))

print("\nFold")
display(compare_summary(calibrated["fold"].astype("Int64")))

cqr_output_path = "../data/interim/mondrian_cqr_groupkfold.csv"
calibrated.to_csv(cqr_output_path, index=True)
print(f"\nSaved calibrated intervals to {cqr_output_path}")


Overall
Calibrated:   coverage=0.8023, median width=0.4021
Uncalibrated: coverage=0.7989, median width=0.3982

TRUE price band


,group,calibrated_coverage,calibrated_width_pct,uncalibrated_coverage,uncalibrated_width_pct,n
0,<80k,0.788491,0.397948,0.792305,0.398967,88346
1,80-150k,0.838950,0.392352,0.839287,0.394064,74126
2,>150k,0.730992,0.468085,0.684271,0.416052,21018



PREDICTED price band


,group,calibrated_coverage,calibrated_width_pct,uncalibrated_coverage,uncalibrated_width_pct,n
0,<80k,0.797742,0.398313,0.800005,0.398155,83077
1,80-150k,0.806932,0.394472,0.812419,0.399470,82002
2,>150k,0.802129,0.462281,0.733800,0.389861,18411



Area band


,group,calibrated_coverage,calibrated_width_pct,uncalibrated_coverage,uncalibrated_width_pct,n
0,<20,0.794151,0.421003,0.799373,0.422976,35424
1,20-30,0.797273,0.358589,0.800556,0.363403,67608
2,30-50,0.813552,0.417950,0.803440,0.411727,49644
3,50-80,0.809181,0.494588,0.795455,0.477776,26009
4,>80,0.779188,0.597631,0.744225,0.548855,4805



Region


,group,calibrated_coverage,calibrated_width_pct,uncalibrated_coverage,uncalibrated_width_pct,n
0,city,0.763216,0.424308,0.766605,0.427522,59320
1,town_village,0.280757,0.572461,0.287066,0.576784,317
2,ward,0.822338,0.391229,0.815693,0.386568,123853



Fold


,group,calibrated_coverage,calibrated_width_pct,uncalibrated_coverage,uncalibrated_width_pct,n
0,0,0.800589,0.407882,0.796665,0.402163,36698
1,1,0.814050,0.411198,0.799063,0.400608,36698
2,2,0.792795,0.401086,0.789689,0.394363,36698
3,3,0.811407,0.396512,0.807455,0.395649,36698
4,4,0.792605,0.391739,0.801679,0.398101,36698



Saved calibrated intervals to ../data/interim/mondrian_cqr_groupkfold.csv


In [13]:

def same_building_comps(df, idx, min_n=3):
    row = df.loc[idx]

    pool = df[
        (df["building_key"] == row["building_key"])
        & (df.index != idx)
        & (df["lookalike_id"] != row["lookalike_id"])
    ].copy()

    pool = pool[
        pool["Price_yen"].notna()
        & pool["area_m2"].notna()
        & (pool["area_m2"] > 0)
    ]

    if len(pool) < min_n or pd.isna(row["area_m2"]) or row["area_m2"] <= 0:
        return None

    rent_per_m2 = pool["Price_yen"] / pool["area_m2"]
    median_ppm2 = rent_per_m2.median()
    p25_ppm2 = rent_per_m2.quantile(0.25)
    p75_ppm2 = rent_per_m2.quantile(0.75)

    return {
        "n": len(pool),
        "median_ppm2": median_ppm2,
        "p25_ppm2": p25_ppm2,
        "p75_ppm2": p75_ppm2,
        "est": median_ppm2 * row["area_m2"],
        "est_lo": p25_ppm2 * row["area_m2"],
        "est_hi": p75_ppm2 * row["area_m2"],
    }


sample_idx = df.sample(
    n=min(5000, len(df)),
    random_state=0
).index

records = []
for idx in sample_idx:
    result = same_building_comps(df, idx)
    if result is not None:
        records.append({
            "idx": idx,
            **result,
            "true_price": df.loc[idx, "Price_yen"],
        })

comps = pd.DataFrame(records)

share_with_result = len(comps) / len(sample_idx)
median_n = comps["n"].median() if len(comps) else np.nan
mape = (
    (comps["est"] - comps["true_price"]).abs()
    / comps["true_price"]
).mean() if len(comps) else np.nan

print(f"Sample size: {len(sample_idx):,}")
print(f"Share with same-building result: {share_with_result:.4f}")
print(f"Median number of comparables: {median_n:.1f}")
print(f"MAPE (in-sample, optimistic): {mape:.4f}")


Sample size: 5,000
Share with same-building result: 0.3440
Median number of comparables: 8.0
MAPE (in-sample, optimistic): 0.0548


In [ ]:
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

# Similar-listing comparables with sklearn NearestNeighbors. Imports: numpy as np, pandas as pd,
# NearestNeighbors, StandardScaler, GroupKFold.
# Features: log(area_m2), age_years, walk_min_nearest, n_rooms, stories_above (fill NaN with
# the training median). Weights: multiply scaled columns by [3, 1, 1, 1.5, 0.5].
# Restrict neighbours to the SAME municipality (loop per municipality).
# Under GroupKFold on group_coarse (reuse splits): for each fold, fit the scaler and
# NearestNeighbors on TRAIN rows of each municipality only, query TEST rows with k=10,
# exclude nothing else (test groups are unseen, so twins are excluded automatically).
# For each test row: neighbours' rent per m2 -> median, p25, p75, scaled by the row's area.
# Store comp_est, comp_lo, comp_hi, comp_n by index label.
# Print: MAE and median % error of comp_est vs true rent, and compare with XGBoost OOF (11,179).
# Also print the share of true rents inside [comp_lo, comp_hi].
# Do not fit on test rows. Do not tune.
feature_names = [
    "log_area_m2",
    "age_years",
    "walk_min_nearest",
    "n_rooms",
    "stories_above",
]
weights = np.array([3.0, 1.0, 1.0, 1.5, 0.5])
k = 10

comp = pd.DataFrame(
    np.nan,
    index=df.index,
    columns=["comp_est", "comp_lo", "comp_hi", "comp_n"],
)

municipalities = (
    df["municipality"]
    .astype("string")
    .fillna("__MISSING_MUNICIPALITY__")
)

def make_features(frame):
    features = pd.DataFrame(index=frame.index)
    area = pd.to_numeric(frame["area_m2"], errors="coerce")
    features["log_area_m2"] = np.where(area > 0, np.log(area), np.nan)
    features["age_years"] = frame["age_years"]
    features["walk_min_nearest"] = frame["walk_min_nearest"]
    features["n_rooms"] = frame["n_rooms"]
    features["stories_above"] = frame["stories_above"]
    return features[feature_names].astype(float)

for tr, te in splits:
    train_frame = df.iloc[tr]
    test_frame = df.iloc[te]

    train_features = make_features(train_frame)
    test_features = make_features(test_frame)

    train_municipalities = municipalities.iloc[tr]
    test_municipalities = municipalities.iloc[te]

    for municipality in test_municipalities.unique():
        train_mask = train_municipalities.to_numpy() == municipality
        test_mask = test_municipalities.to_numpy() == municipality

        train_positions = np.flatnonzero(train_mask)
        test_positions = np.flatnonzero(test_mask)

        if len(train_positions) == 0:
            continue

        x_train = train_features.iloc[train_positions].copy()
        x_test = test_features.iloc[test_positions].copy()

        medians = x_train.median()
        x_train = x_train.fillna(medians)
        x_test = x_test.fillna(medians)

        valid_test = x_test.notna().all(axis=1).to_numpy()
        if not valid_test.any():
            continue

        scaler = StandardScaler()
        x_train_scaled = scaler.fit_transform(x_train) * weights
        x_test_scaled = scaler.transform(x_test.loc[valid_test]) * weights

        n_neighbors = min(k, len(x_train_scaled))
        nn = NearestNeighbors(n_neighbors=n_neighbors)
        nn.fit(x_train_scaled)

        _, neighbor_indices = nn.kneighbors(x_test_scaled)

        train_indices = train_frame.index[train_positions]
        train_areas = train_frame.loc[train_indices, "area_m2"].to_numpy()
        train_prices = train_frame.loc[train_indices, "Price_yen"].to_numpy()

        valid_train = (
            np.isfinite(train_areas)
            & (train_areas > 0)
            & np.isfinite(train_prices)
        )

        neighbor_prices = train_prices[neighbor_indices]
        neighbor_areas = train_areas[neighbor_indices]

        valid_neighbors = (
            valid_train[neighbor_indices]
            & np.isfinite(neighbor_prices)
            & np.isfinite(neighbor_areas)
            & (neighbor_areas > 0)
        )

        ppm2 = np.divide(
            neighbor_prices,
            neighbor_areas,
            out=np.full_like(neighbor_prices, np.nan, dtype=float),
            where=valid_neighbors,
        )

        row_area = test_frame.iloc[test_positions].loc[
            x_test.index[valid_test], "area_m2"
        ].to_numpy()

        enough = np.isfinite(ppm2).sum(axis=1) > 0
        if not enough.any():
            continue

        estimates = np.nanmedian(ppm2, axis=1) * row_area
        lower = np.nanpercentile(ppm2, 25, axis=1) * row_area
        upper = np.nanpercentile(ppm2, 75, axis=1) * row_area
        counts = np.isfinite(ppm2).sum(axis=1)

        result_indices = x_test.index[valid_test][enough]
        comp.loc[result_indices, "comp_est"] = estimates[enough]
        comp.loc[result_indices, "comp_lo"] = lower[enough]
        comp.loc[result_indices, "comp_hi"] = upper[enough]
        comp.loc[result_indices, "comp_n"] = counts[enough]

valid = comp["comp_est"].notna() & df["Price_yen"].notna()
true_price = df.loc[valid, "Price_yen"]
comp_est = comp.loc[valid, "comp_est"]

comp_mae = (comp_est - true_price).abs().mean()
comp_median_pct_error = (
    (comp_est - true_price).abs() / true_price
).median()

xgb_valid = valid & oof.notna()
xgb_mae = (oof.loc[xgb_valid] - df.loc[xgb_valid, "Price_yen"]).abs().mean()

inside = (
    valid
    & comp["comp_lo"].notna()
    & comp["comp_hi"].notna()
    & df["Price_yen"].between(comp["comp_lo"], comp["comp_hi"])
)

print(f"Comparable rows: {valid.sum():,} / {len(df):,}")
print(f"Comparables MAE: {comp_mae:,.2f}")
print(f"Comparables median % error: {comp_median_pct_error:.4f}")
print(f"XGBoost OOF MAE: {xgb_mae:,.2f}")
print(f"True prices inside [comp_lo, comp_hi]: {inside.sum() / valid.sum():.4f}")